# *Psi4* Workflow

:::{div}
:class: colab-badge
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blinkletter/UsingPsi4/blob/main/book/05-RotationProfiles/052-Workflow.ipynb)
:::

Here will be presented the **step by step** workflow for calculating the energy profile for rotating the central bond of butane. We have seen that butane has energy minima at torsion angles of 180&deg; and about 67&deg;. It also has a maximum at 0&deg;. Is the other maximum at 120&deg; or is it slightly **offset** due to the size difference between a hydrogen and a methyl group? We will make many molecules with locked torsion angles varying from 0&deg; to 180&deg; and then calculate the **optimized** structures at each locked angle. The energies will be collected as we go and saved to a file, and in the [next section](053-WorkflowData.ipynb) we will plot the potential energy profile for bond rotation.

In the *GAMESS* book, this took three tools: a *Unix* script using *sed* to write nineteen input files, the *GamessQ* program to run them, and another script using *grep* to dig the energies out of nineteen log files. Here, it takes one *Python* **loop**.

In [1]:
####################################################
### Import Libraries and set up global variables ###
####################################################

from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Install packages only if needed
# Google Colab does not include every Python package
#
# Psi4 cannot be installed with pip. On Colab we first install conda
# (condacolab), which RESTARTS the session. After the restart, run
# this cell again and Psi4 will be installed with conda.
try:                  # if the package exists import it, otherwise install it
    import psi4
except ImportError:
    if IN_COLAB:
        !pip -q install condacolab
        import condacolab
        condacolab.install()
        !mamba install -q -y -c conda-forge psi4
    import psi4

try:
    import py3Dmol
except ImportError:
    !pip -q install py3Dmol
    import py3Dmol

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Plot style sheets from the book's GitHub repository
if IN_COLAB and not Path("styles/tufte.mplstyle").exists():
    !wget -q -P styles https://raw.githubusercontent.com/blinkletter/UsingPsi4/main/styles/tufte.mplstyle
    !wget -q -P styles https://raw.githubusercontent.com/blinkletter/UsingPsi4/main/styles/S2_classic2.mplstyle

if IN_COLAB:
    plt.style.use("styles/tufte.mplstyle")
else:
    plt.style.use("../../styles/tufte.mplstyle")

## Setting Up the Notebook

As in the previous chapters, the hidden **first code cell** imports *Psi4*, *py3Dmol*, *NumPy*, *pandas* and *Matplotlib* and applies the plot **style sheet**. You will see it when you open the notebook in *Jupyter*, *Visual Studio Code* or *Google Colab*.

```{admonition} Running this notebook in Google Colab
:class: dropdown
The first code cell checks whether the notebook is running in *Google Colab*. If it is, the cell **installs** *Psi4* for you. *Colab* does not come with *conda*, so the cell first installs it using a package called *condacolab*. This **restarts** the *Colab* session. When it has restarted, simply **run the first cell again**. It will then install *Psi4* with *conda* (*Psi4* cannot be installed with *pip*), install *py3Dmol* with *pip*, and download the plot style sheets from the book's *GitHub* repository. This takes a few minutes, and you must do it every time you start a new *Colab* session. On your own computer, the cell just imports the packages.
```

Next, we tell *Psi4* how much of the computer it may use, and set the options that we will need in this chapter.

In [2]:
psi4.set_memory("4 GB")
psi4.set_num_threads(4)

psi4.set_options({"scf_type": "pk",
                  "frozen_dihedral": "4 1 2 3",
                  "geom_maxiter": 100})


  Memory set to   3.725 GiB by Python driver.
  Threads set to 4 by Python driver.


| Line | Notes |
| :---- | :---- |
| `"frozen_dihedral": "4 1 2 3"` | **Freeze** the central C4&ndash;C1&ndash;C2&ndash;C3 torsion (atoms counted from one) for every optimization in this notebook. This is `IFREEZ(1)=6` in the *GAMESS* input file. |
| `"geom_maxiter": 100` | Allow up to 100 optimization steps (the default is 50). The *GAMESS* scan used `NSTEP=1000`, and some of its jobs needed hundreds of steps. We will see whether *Psi4* needs that many. |

### Some Helpful Tools

We only need two of our helper functions here. In the webbook, click on the code section to expand it.

In [3]:
def dihedral(mol, i, j, k, l):
    xyz = mol.geometry().np
    b1 = xyz[j] - xyz[i]
    b2 = xyz[k] - xyz[j]
    b3 = xyz[l] - xyz[k]
    n1 = np.cross(b1, b2)
    n2 = np.cross(b2, b3)
    x = np.dot(n1, n2)
    y = np.dot(np.cross(n1, n2), b2) / np.linalg.norm(b2)
    return np.degrees(np.arctan2(y, x))

def search_file(filename, text):
    with open(filename) as f:
        for number, line in enumerate(f, start=1):
            if text.lower() in line.lower():
                print(f"{number:6d}: {line.rstrip()}")

| Function | What it does |
| :---- | :---- |
| `dihedral(mol, i, j, k, l)` | The **torsion** angle `i`&ndash;`j`&ndash;`k`&ndash;`l` in degrees (atoms counted from zero). |
| `search_file(filename, text)` | **Prints** every line of a text file that contains `text`, with its line number. |

## The Molecule

First we **write** a *Z-matrix* for butane. We will not use symmetry because we want to rotate the bond to many different angles. We will use the *Z-matrix* method because it will identify the central **torsion** as a coordinate. Then we can **lock** that coordinate.

In [4]:
def make_butane(torsion):
    return psi4.geometry(f"""
0 1
C
C   1    1.54000
C   2    1.54000  1  109.0
C   1    1.54000  2  109.4712  3  {torsion}
H   1    1.14000  2  109.4712  4 -120.0000
H   1    1.14000  5  108.5      2  120.0000
H   2    1.14000  1  109.4712  3 -120.0000
H   2    1.14000  7  109.4712  1  120.0000
H   3    1.14000  2  109.4712  8   60.0000
H   3    1.14000  9  109.4712  2  120.0000
H   3    1.14000  9  109.4712  10 120.0000
H   4    1.14000  1  109.0      6   60.0000
H   4    1.14000  12 108.5      1  120.0000
H   4    1.14000  12 109.4712  13 119.0000
symmetry c1
""")

You will see that not all angles are 109.47&deg;. In the *GAMESS* book I **jiggled** some to avoid any perfect symmetry locks that prevented the energy minimization from converging in one of the jobs. We will keep the same jiggles so that we start from the same place. The dihedral angles for hydrogens 9 and 12 are 60&deg; rather than 0&deg; so that the methyl groups start **staggered**, as discussed in the [previous section](051-RotationProfiles.ipynb).

## Making the Series

In the *GAMESS* book, a *bash* script used the *Unix* tools *seq* and *sed* to replace the text `NUMBER` in the input file with each torsion angle, writing a new input file each time. In *Python* we simply make a **list** of the angles.

In [5]:
torsions = np.arange(0, 181, 10)
print(torsions)

[  0  10  20  30  40  50  60  70  80  90 100 110 120 130 140 150 160 170
 180]


| Line | Notes |
| :---- | :---- |
| `np.arange(0, 181, 10)` | The numbers from 0 **up to, but not including,** 181 in steps of 10. This is `seq -w 000 010 180` in the *bash* script. |

Nineteen angles with values for the 4,1,2,3 torsion from 0 to 180 degrees.

## Running the Jobs

Now the **loop**. For each angle we build a molecule, give it its own output file, optimize it with the torsion frozen, and record the results in a list. Each output file goes into a folder named `scan_HF321G` so that the chapter folder does not drown in files. Brew a coffee. This will take a few minutes.

In [6]:
Path("scan_HF321G").mkdir(exist_ok=True)

results = []
molecules = {}

for phi in torsions:
    mol = make_butane(phi)
    psi4.set_output_file(f"scan_HF321G/butane_scan_{phi:03d}.out", False)

    try:
        E, history = psi4.optimize("hf/3-21g", molecule=mol, return_history=True)
        steps = len(history["energy"])
        max_grad = np.max(np.abs(history["gradient"][-1]))
    except psi4.OptimizationConvergenceError:
        E, steps, max_grad = np.nan, 100, np.nan

    results.append({"Torsion": int(phi),
                    "Final torsion": abs(dihedral(mol, 3, 0, 1, 2)),
                    "Energy": E,
                    "Steps": steps,
                    "Max gradient": max_grad})
    molecules[phi] = mol
    psi4.core.clean()
    print(f"{phi:4d}  {E:.10f}  {steps:3d} steps")

Optimizer: Optimization complete!
   0  -156.4228464782   10 steps
Optimizer: Optimization complete!
  10  -156.4233726902   10 steps
Optimizer: Optimization complete!
  20  -156.4247889217    9 steps
Optimizer: Optimization complete!
  30  -156.4267021623    9 steps
Optimizer: Optimization complete!
  40  -156.4286426645    8 steps
Optimizer: Optimization complete!
  50  -156.4301846974    8 steps
Optimizer: Optimization complete!
  60  -156.4310615500    7 steps
Optimizer: Optimization complete!
  70  -156.4312168286    6 steps
Optimizer: Optimization complete!
  80  -156.4306888494    5 steps
Optimizer: Optimization complete!
  90  -156.4295738497    5 steps
Optimizer: Optimization complete!
 100  -156.4282208760    5 steps
Optimizer: Optimization complete!
 110  -156.4271414172    5 steps
Optimizer: Optimization complete!
 120  -156.4267262681    5 steps
Optimizer: Optimization complete!
 130  -156.4271134847    6 steps
Optimizer: Optimization complete!
 140  -156.4281851170    5 s

| Line | Notes |
| :---- | :---- |
| `results = []` | An empty **list**. We will add one item to it for each calculation. |
| `molecules = {}` | An empty **dictionary**. We will keep each optimized molecule in it, labelled by its torsion angle, so we can use the structures again later. |
| `for phi in torsions:` | Repeat the indented lines for **each** angle in the list. |
| `f"scan_HF321G/butane_scan_{phi:03d}.out"` | A new output file for each job. `{phi:03d}` writes the angle with three digits (`000`, `010`, &hellip; `180`) so that the files sort in order. This is what `seq -w` did for the *GAMESS* file names. |
| `try:` &hellip; `except psi4.OptimizationConvergenceError:` | If an optimization runs out of steps, *Psi4* raises an **error** and would stop the whole loop. The `except` line catches that error, records *NaN* ("not a number") for the energy and carries on with the next angle. |
| `history["gradient"][-1]` | The gradient at the **last** step. `np.max(np.abs(...))` gives its largest component, like `GRAD. MAX` in *GAMESS*. |
| `results.append({...})` | Adds a **dictionary** of results for this angle to the list. |
| `abs(dihedral(mol, 3, 0, 1, 2))` | The torsion **after** the optimization, to check that it really was frozen. |
| `psi4.core.clean()` | Cleans up *Psi4*'s scratch files before the next job. |

Nineteen jobs, one cell, no files to edit. The final column printed is the number of optimization **steps** each job needed.

## Extracting the Results

In the *GAMESS* book, a second script used *grep* and *tail* to pull the last `NSERCH` line from each of the nineteen log files. We already have the results in the `results` list. Let us turn it into a *pandas* table and **save** it as a JSON file for the next section.

In [7]:
scan_321g = pd.DataFrame(results)
scan_321g.to_json("butane_scan_HF321G.json", orient="records", indent=2)
scan_321g

,Torsion,Final torsion,Energy,Steps,Max gradient
0,0,0.000430,-156.422846,10,0.000021
1,10,10.000000,-156.423373,10,0.000015
2,20,20.000000,-156.424789,9,0.000027
3,30,30.000000,-156.426702,9,0.000022
4,40,40.000000,-156.428643,8,0.000056
5,50,50.000000,-156.430185,8,0.000037
6,60,60.000000,-156.431062,7,0.000028
7,70,70.000000,-156.431217,6,0.000067
8,80,80.000000,-156.430689,5,0.000078
9,90,90.000000,-156.429574,5,0.000057


| Line | Notes |
| :---- | :---- |
| `pd.DataFrame(results)` | A list of dictionaries makes a tidy **table**: each dictionary becomes a row, and the keys become the column headings. |
| `to_json("butane_scan_HF321G.json", ...)` | Saves the table beside this notebook. |

Of course, the output files are all still there if we want them. Here is the *Python* version of the *GAMESS* `extract.sh` script. It prints the **last** energy line from each output file.

In [8]:
for name in sorted(Path("scan_HF321G").glob("butane_scan_*.out")):
    with open(name) as f:
        energy_lines = [line for line in f if "@RHF Final Energy" in line]
    print(name.name, energy_lines[-1].strip(), f"({len(energy_lines)} steps)")

butane_scan_000.out @RHF Final Energy:  -156.42284647824391 (10 steps)
butane_scan_010.out @RHF Final Energy:  -156.42337269019637 (10 steps)
butane_scan_020.out @RHF Final Energy:  -156.42478892172514 (9 steps)
butane_scan_030.out @RHF Final Energy:  -156.42670216226637 (9 steps)
butane_scan_040.out @RHF Final Energy:  -156.42864266448922 (8 steps)
butane_scan_050.out @RHF Final Energy:  -156.43018469744118 (8 steps)
butane_scan_060.out @RHF Final Energy:  -156.43106155000126 (7 steps)
butane_scan_070.out @RHF Final Energy:  -156.43121682864745 (6 steps)
butane_scan_080.out @RHF Final Energy:  -156.43068884935158 (5 steps)
butane_scan_090.out @RHF Final Energy:  -156.42957384971263 (5 steps)
butane_scan_100.out @RHF Final Energy:  -156.42822087597861 (5 steps)
butane_scan_110.out @RHF Final Energy:  -156.42714141715328 (5 steps)
butane_scan_120.out @RHF Final Energy:  -156.42672626806407 (5 steps)
butane_scan_130.out @RHF Final Energy:  -156.42711348470129 (6 steps)
butane_scan_140.ou

| Line | Notes |
| :---- | :---- |
| `sorted(Path(...).glob("butane_scan_*.out"))` | All the output files in the folder whose names match the **pattern**, in order. `*` matches anything. |
| `[line for line in f if ...]` | A list of only the lines that contain `@RHF Final Energy`. This is our *grep*. |
| `energy_lines[-1]` | The **last** of those lines. This is our *tail*. |

## Inspecting the Result

**Look** at the `Steps` column. Every job converged, in 4 to 10 steps. None came close to running out of steps. The torsion in the `Final torsion` column is equal to the target in every job, apart from tiny drifts at 0&deg; (0.0004&deg;) and 180&deg; (179.9993&deg;).

In the *GAMESS* scan, all but three jobs converged in less than 30 steps. The jobs at 100&deg;, 130&deg; and 160&deg; needed 390, 362 and 646 steps. (In an earlier attempt, the job at 100&deg; ran out of steps after 1000!) All three eventually found their way home, but only because this was a light-weight theory level and I was confident in **allowing** for a large number of steps. Why did they take so long? **Bad luck** with the exact structure of the input started them on the long path. *Psi4*'s optimizer, working in redundant internal coordinates, had no such trouble. Its slowest jobs, at 0&deg; and 10&deg;, needed 10 steps, and every energy agrees with the *GAMESS* scan to better than 4 &times; 10<sup>&minus;7</sup> Hartree. Clearly, **how you start matters**, and so does the optimizer. For high levels of theory it is important to start as close to the final answer as possible.

-----

## Higher Level Theory

For fun, let us repeat the scan at the HF/6-311++G(d,p) level of theory, the level that we used for methane and ethane. This basis set has **many** more functions than 3-21G, and each job will take much longer. But we have a trick. Our HF/3-21G structures are **excellent starting points**. We kept every optimized molecule in the `molecules` dictionary, so we can start each new job from the HF/3-21G structure rather than from our rough *Z-matrix*. The torsion is still frozen at the same value.

```{note}
This loop takes about 9 minutes on my laptop. If you are short of time, skip it. The next section will load the saved results from the book's *GitHub* repository instead.
```

In [9]:
Path("scan_HF6311ppGdp").mkdir(exist_ok=True)

results = []

for phi in torsions:
    mol = molecules[phi]
    psi4.set_output_file(f"scan_HF6311ppGdp/butane_scan_{phi:03d}.out", False)

    try:
        E, history = psi4.optimize("hf/6-311++g(d,p)", molecule=mol, return_history=True)
        steps = len(history["energy"])
        max_grad = np.max(np.abs(history["gradient"][-1]))
    except psi4.OptimizationConvergenceError:
        E, steps, max_grad = np.nan, 100, np.nan

    results.append({"Torsion": int(phi),
                    "Final torsion": abs(dihedral(mol, 3, 0, 1, 2)),
                    "Energy": E,
                    "Steps": steps,
                    "Max gradient": max_grad})
    psi4.core.clean()
    print(f"{phi:4d}  {E:.10f}  {steps:3d} steps")

scan_6311 = pd.DataFrame(results)
scan_6311.to_json("butane_scan_HF6311ppGdp.json", orient="records", indent=2)

Optimizer: Optimization complete!
   0  -157.3303330669    4 steps
Optimizer: Optimization complete!
  10  -157.3309332222    4 steps
Optimizer: Optimization complete!
  20  -157.3324970202    5 steps
Optimizer: Optimization complete!
  30  -157.3345249205    5 steps
Optimizer: Optimization complete!
  40  -157.3364986282    5 steps
Optimizer: Optimization complete!
  50  -157.3379945568    4 steps
Optimizer: Optimization complete!
  60  -157.3387729003    4 steps
Optimizer: Optimization complete!
  70  -157.3388158533    4 steps
Optimizer: Optimization complete!
  80  -157.3382358235    4 steps
Optimizer: Optimization complete!
  90  -157.3371813627    4 steps
Optimizer: Optimization complete!
 100  -157.3359455660    4 steps
Optimizer: Optimization complete!
 110  -157.3349726713    4 steps
Optimizer: Optimization complete!
 120  -157.3346411919    4 steps
Optimizer: Optimization complete!
 130  -157.3350961397    3 steps
Optimizer: Optimization complete!
 140  -157.3362143036    4 s

| Line | Notes |
| :---- | :---- |
| `mol = molecules[phi]` | Instead of building a new molecule, we take the **optimized** HF/3-21G molecule for this angle. *Psi4* will optimize it **again** at the higher level, and update it in place. |
| `"hf/6-311++g(d,p)"` | A triple-split valence basis set with **diffuse** functions (`++`) and **polarization** functions (`(d,p)`). This is `GBASIS=N311 NGAUSS=6 NDFUNC=1 NPFUNC=1 DIFFSP=.TRUE. DIFFS=.TRUE.` in *GAMESS*. |

Starting from the HF/3-21G structures, each job needed only 3 to 5 steps. The cheap calculation did most of the work for the expensive one. We stand on the shoulders of giants.

We now have the raw data for both levels of theory. Next we will process it and **plot the results** using *Python*. The [next section](053-WorkflowData.ipynb) is an **interactive *Python*** notebook that details this workflow.

```{warning} AI Statement
:class: dropdown
This page is an adaptation of the "*GAMESS* Workflow" section of my [previous web book on using *GAMESS*](https://blinkletter.github.io/UsingGAMESS). About **a quarter** of the text is **my own work**, carried over word for word or lightly edited, including the introduction, the jiggled *Z-matrix* and the discussion of the slow jobs. The **rest of the page** was written by an **LLM system** at my request. I asked it to replace the *Unix* scripts and *GamessQ* with a *Python* loop for *Psi4*, and to add a second scan at a higher level of theory starting from the HF/3-21G structures. All of the *Python* code, the line-by-line explanations of it and the commentary on the results were written by the LLM. I ran every calculation on my own computer, and the numbers quoted in the text come from those results.
```